# 99 Final: the chosen model against matrix pricing

The submission notebook. It runs top to bottom from the panel and produces the headline table and the quality
gate verdict.

| Mode | `OPEN_TEST` | Fits on | Scores on | When |
|---|---|---|---|---|
| Dry run | `False` | train | valid | Any time before Saturday: checks the whole pipeline without touching test |
| Final | `True` | train + valid | test (once) | Saturday morning, after the choice is logged in `RUN_LOG.md` (P13) |

The gate (`config.yaml`, `quality_gate`): at least 10% lower MAE than matrix pricing re-marked on each snapshot
date, with a 95% issuer-bootstrap interval above zero, in IG and in HY.

`OPEN_TEST = True` writes `notebooks/test_opened.json` with the model's fingerprint (every setting, the feature
list, the fitting months and the data checksum). Commit and push it straight away so the whole group shares it.
Re-running the same model is fine; running a different model against test stops with an error.

In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor

from ml2.baseline import fit_matrix_frozen, matrix_same_date, median_baseline, predict_matrix_frozen
from ml2.config import load_config, resolve
from ml2.data import check_panel, load_panel
from ml2.evaluate import gate_report, metrics_by_stratum
from ml2.features import make_preprocessor, xy
from ml2.final import model_spec, open_test_once

cfg = load_config()
TARGET = cfg["problem"]["target"]

OPEN_TEST = False  # True only on Saturday morning, once the choice is in RUN_LOG.md

## 1. The chosen model

Filled in on Saturday from `RUN_LOG.md` (P13): the best model on validation, with the settings frozen on Friday
evening. Until then a pruned regression tree stands in, so the notebook can be dry-run end to end.

In [ ]:
FINAL_NAME = "PLACEHOLDER: regression tree (replace on Saturday)"


def make_final_model():
    return Pipeline([
        ("pre", make_preprocessor(scale=False, one_hot=False)),
        ("est", DecisionTreeRegressor(max_depth=8, min_samples_leaf=50, random_state=42)),
    ])


SPEC = model_spec(FINAL_NAME, make_final_model(), ["train", "valid"])
print(SPEC[:300], "...")

## 2. Data

In [ ]:
check_panel()
if OPEN_TEST:
    print(open_test_once(SPEC))  # before test is loaded: records or checks the model
fit_parts, score_part = (["train", "valid"], "test") if OPEN_TEST else (["train"], "valid")

panel = load_panel(partitions=fit_parts + [score_part], include_test=OPEN_TEST)
fit = panel[panel["partition"].isin(fit_parts)]
score = panel[(panel["partition"] == score_part) & panel[TARGET].notna()].copy()
print(f"Fit on {fit_parts}: {len(fit):,} rows. Score on {score_part}: {len(score):,} rows")

## 3. Fit and predict

In [ ]:
X_fit, y_fit = xy(fit)
X_score, _ = xy(score)

model = make_final_model().fit(X_fit, y_fit)
score["pred_final"] = model.predict(X_score)

## 4. Baselines

Matrix pricing re-marked on each date is the gate baseline: each bond priced from the same-day spreads of other
companies' bonds in its grade x maturity x sector cell. The median and frozen matrix are reference rows.

In [ ]:
score["pred_matrix_same_date"] = matrix_same_date(score)
score["pred_matrix_frozen"] = predict_matrix_frozen(fit_matrix_frozen(fit), score)
score["pred_median"] = median_baseline(fit, score)

## 5. Headline table: MAE in bp by grade

In [ ]:
ROWS = {
    "pred_final": FINAL_NAME,
    "pred_matrix_same_date": "Matrix pricing, re-marked each date (gate baseline)",
    "pred_matrix_frozen": "Matrix pricing, frozen at the fitting months",
    "pred_median": "Median spread",
}
headline = pd.concat(
    [metrics_by_stratum(score, TARGET, col, "grade_bucket").assign(model=name) for col, name in ROWS.items()]
)
table = headline.pivot(index="model", columns="stratum", values="mae_bps").loc[list(ROWS.values())]
table = table[["IG", "HY", "ALL"]].round(1)
table

## 6. Quality gate

In [ ]:
ok, gate = gate_report(score, TARGET, "pred_final", "pred_matrix_same_date", cfg["quality_gate"])
print(f"{score_part}: gate {'PASSED' if ok else 'NOT PASSED'}")
gate.round(3)

In [ ]:
mode = "final_test" if OPEN_TEST else "dryrun_valid"
out = resolve("reports")
table.to_csv(out / f"headline_{mode}.csv")
gate.to_csv(out / f"gate_{mode}.csv")
print(f"Saved reports/headline_{mode}.csv and reports/gate_{mode}.csv. Add a line to notebooks/RUN_LOG.md.")